In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
import palettable
import os
import numpy as np
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from datetime import datetime

from machine_intelligence.utilities.viz_setup import figure_setup_grid, VariableNames

from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
import matplotlib.ticker as ticker
import random

from machine_intelligence.models.GatedTransformer import GatedTransformer
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule



In [ ]:

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime('%Y-%m-%d')
plot_output_dir = project_root / 'plot_outputs' / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)
# model_path = project_root / Path("experiment_logs/gtn-debug/version_118/checkpoints/epoch=5-step=186.ckpt")

In [ ]:

model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final-3/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(model_path, map_location="cpu")


In [ ]:

wandb.init(mode="disabled")


In [ ]:

os.environ['PYTHONHASHSEED'] = str(42)
lit.seed_everything(42, workers=True)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.benchmark = False



In [ ]:
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_prev_0.3_imp_none"])

    print(cfg)


In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)
sepsis_pred_data.setup(stage="test")
inference_dl = sepsis_pred_data.test_dataloader()

batch = next(iter(inference_dl))
batch_features_ts, batch_features_stat, batch_labels = batch


In [ ]:
sample_index = 10

sample_ts = batch_features_ts[sample_index].unsqueeze(0)
sample_stat = batch_features_stat[sample_index].unsqueeze(0)

model.to("cpu")
model.eval()
with torch.no_grad():
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sample_ts, sample_stat, stage="test")


In [ ]:
pred

In [ ]:
len(score_input)

In [ ]:
score_input[0].shape

The gtn has only one layer so we can access it directly the outputs from the forward pass above!

In [ ]:
model.score_input_list[0].shape

### Determining the Time Series Data where data is present

In [ ]:
df_sample_ts = pd.DataFrame(sample_ts.squeeze(0), columns=sepsis_pred_data.considered_features)

# df_sample_ts = VariableNames.get_pretty_dataframe(df=df_sample_ts)
# df_sample_ts = VariableNames.sort_dataframe(df=df_sample_ts)

zero_columns = df_sample_ts.loc[:, (df_sample_ts == 0).all()].columns.to_list()
df_sample_ts_present = df_sample_ts.drop(columns=zero_columns)
df_sample_ts_present.head()

### Sample Time Series Data

In [ ]:

lab_feature_names = VariableNames.get_lab_related_names()
vitals_names = VariableNames.get_vital_sign_names()
lab_feature_names.values()

In [ ]:

fig, axs, font_labels, font_annotations = figure_setup(2, 1, width_cm=14, height_cm=20, num_colors=11,
                                                       palette=palettable.matplotlib.Plasma_20.mpl_colors[:18]
                                                       )

df_sample_ts_present_pretty = VariableNames.get_pretty_dataframe(df_sample_ts_present)



ax0 = axs[0]
ax1 = axs[1]
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', 'h', '*', '+', 'x', 'X', 'd']
line_styles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--", "-.", ":", ]

j = 0
for col in df_sample_ts_present_pretty.columns:
    # if "labname" in col:
    if col in lab_feature_names.values():
        marker = random.choice(markers)
        sns.scatterplot(df_sample_ts_present_pretty[col], ax=ax1, label=col, marker=marker, s=10, zorder=100,
                        alpha=0.8)
    else:
        
        sns.lineplot(df_sample_ts_present_pretty[col], ax=ax0, label=col, linestyle=line_styles[j % len(line_styles)])
        j += 1

# for i, col in enumerate(vitals_names):
#     ax0.plot(df_sample_ts_present,
#                         linestyle=line_styles[i % len(line_styles)],
#                         label=col)

ax0.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annotations,
)
ax1.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annotations,
)

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
ax0.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax0.xaxis.set_major_formatter(five_min_hour_formatter)

ax1.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax1.xaxis.set_major_formatter(five_min_hour_formatter)

ax0.set_xlabel("Time [h]")
ax1.set_xlabel("Time [h]")

ax0.set_ylabel("Dense \n Time Series Features",
               )

ax1.set_ylabel("Sparse \n Time Series Features",
               )

ax0.margins(x=0)
ax1.margins(x=0)

# Adjust layout
plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_sample_data.svg"), format="svg")


## Individual Attention Heads Heatmaps

In [ ]:
fig, gs, axs, font_labels, font_annots = figure_setup_grid(2, 4, width_ratios=[0.9, 0.9, 0.9, 1.1],
                                                           height_ratios=[1, 1], height_cm=10)

# sns.heatmap(score_input_sm[0], ax=axs[0, 0], cbar=False, square=True)
# sns.heatmap(score_input_sm[1], ax=axs[0, 1], cbar=False, square=True)
# sns.heatmap(score_input_sm[2], ax=axs[0, 2], cbar=False, square=True)
# sns.heatmap(score_input_sm[3], ax=axs[0, 3], square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75})

sns.heatmap(score_input_sm[0], ax=axs[0, 0], cbar=False, square=True,   cmap=sns.color_palette("magma", as_cmap=True))
sns.heatmap(score_input_sm[1], ax=axs[0, 1], cbar=False, square=True,   cmap=sns.color_palette("magma", as_cmap=True))
sns.heatmap(score_input_sm[2], ax=axs[0, 2], cbar=False, square=True,   cmap=sns.color_palette("magma", as_cmap=True))
sns.heatmap(score_input_sm[3], ax=axs[0, 3], square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75},   cmap=sns.color_palette("magma", as_cmap=True))

sns.heatmap(score_input_sm[4], ax=axs[1, 0], cbar=False, square=True,   cmap=sns.color_palette("magma", as_cmap=True))
sns.heatmap(score_input_sm[5], ax=axs[1, 1], cbar=False, square=True,   cmap=sns.color_palette("magma", as_cmap=True))
sns.heatmap(score_input_sm[6], ax=axs[1, 2], cbar=False, square=True,   cmap=sns.color_palette("magma", as_cmap=True))
sns.heatmap(score_input_sm[7], ax=axs[1, 3], square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75},   cmap=sns.color_palette("magma", as_cmap=True))

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
for ax in axs[0, 1:]:
    ax.set_yticks([])

for ax in axs[1, 1:]:
    ax.set_yticks([])

for ax in axs[0, :]:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax.xaxis.set_major_formatter(five_min_hour_formatter)
    ax.set_xlabel("Time [h]")

for ax in axs[1, :]:
    ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax.xaxis.set_major_formatter(five_min_hour_formatter)
    ax.set_xlabel("Time [h]")

for i, ax in enumerate(axs[0, :]):
    ax.set_title(f"Head {i + 1}")

for i, ax in enumerate(axs[1, :]):
    ax.set_title(f"Head {i + 5}")

axs[0, 0].yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs[0, 0].yaxis.set_major_formatter(five_min_hour_formatter)
axs[0, 0].set_xlabel("Time [h]")
axs[0, 0].set_ylabel("Time [h]")

axs[1, 0].yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs[1, 0].yaxis.set_major_formatter(five_min_hour_formatter)
axs[1, 0].set_xlabel("Time [h]")
axs[1, 0].set_ylabel("Time [h]")

# axs[0,0].invert_yaxis()
# axs[0,1].invert_yaxis()
# axs[0,2].invert_yaxis()
# axs[0,3].invert_yaxis()

# plt.tight_layout()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_individual_attention_heads.svg"),
                format="svg")


    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_individual_attention_heads.png"),
                format="png", dpi=300)



### Head average sequence level attention unnormalized

In [ ]:
score_sequence_mean = torch.mean(score_input, dim=0)

fig_seq_attn_mean, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=8, height_cm=8, num_colors=11,
                                                                     height_ratios=[1])

sns.heatmap(score_sequence_mean, ax=axs, square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75},   cmap=sns.color_palette("magma", as_cmap=True))

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
# for ax in axs:
#     ax.set_yticks([])

axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")

# for i, ax in enumerate(axs[0,:]):
#     ax.set_title(f"Head {i + 1}")

axs.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.yaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")
axs.set_ylabel("Time [h]")

### Head average sequence level attention softmaxed

In [ ]:
score_sequence_mean = torch.mean(score_input_sm, dim=0)

fig_seq_attn_mean, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=8, height_cm=8, num_colors=11,
                                                                     height_ratios=[1])

sns.heatmap(score_sequence_mean, ax=axs, square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75},
              cmap=sns.color_palette("magma", as_cmap=True))

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
# for ax in axs:
#     ax.set_yticks([])

axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")

# for i, ax in enumerate(axs[0,:]):
#     ax.set_title(f"Head {i + 1}")

axs.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.yaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")
axs.set_ylabel("Time [h]")

if save_as_svg := True:
    fig_seq_attn_mean.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_temporal_averaged.svg"),
                              format="svg")

    fig_seq_attn_mean.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_temporal_averaged.png"),
                              format="png", dpi=300)
    

### Softmaxed Feature Level Attention Averaged across heads

In [ ]:
df_score_channel_mean = pd.DataFrame(data=torch.mean(score_channel_sm, dim=0).numpy(),
                                     columns=sepsis_pred_data.considered_features,
                                     index=sepsis_pred_data.considered_features)

df_score_channel_mean = VariableNames.get_pretty_dataframe(df=df_score_channel_mean)
df_score_channel_mean = VariableNames.get_human_readable_index(df=df_score_channel_mean)
df_score_channel_mean = VariableNames.sort_dataframe(df=df_score_channel_mean)

fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)

# sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
# sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
sns.heatmap(df_score_channel_mean, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
            xticklabels=1, yticklabels=1,   cmap=sns.color_palette("magma", as_cmap=True))

# axs.invert_xaxis()
# axs.invert_yaxis()

axs.set_xlabel("Time Series Features",
               )

axs.set_ylabel("Time Series Features",
               )

axs.set_xticklabels(axs.get_xticklabels(), fontsize=3, rotation=80, ha="right")
axs.set_yticklabels(axs.get_yticklabels(), fontsize=3)

# Adjust layout
plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_features_averaged.svg"), format="svg")

    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_features_averaged.png"), format="png", dpi=300)


In [ ]:
attention_score_channel = torch.mean(score_channel_sm, dim=0).numpy()
attention_score_ts = torch.mean(score_input_sm, dim=0).numpy()

attention_score_ts_mean_query = np.mean(attention_score_ts, axis=0)
attention_score_ts_mean_key = np.mean(attention_score_ts, axis=1)

attention_score_channel_mean_query = np.mean(attention_score_channel, axis=0)
attention_score_channel_mean_key = np.mean(attention_score_channel, axis=1)


### Temporal Attention averaged across query 

In [ ]:
sns.heatmap(np.expand_dims(attention_score_ts_mean_query, 0), square=False,   cmap=sns.color_palette("magma", as_cmap=True))


In [ ]:
df_attention_score_ts_mean_query = pd.DataFrame(data=attention_score_ts_mean_query, columns=["Average Temporal Attention Score"])

In [ ]:
fig_seq_attn_mean_line, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=14, height_cm=7, num_colors=11,)

color_palette = palettable.tableau.PurpleGray_6.mpl_colors[0:]

sns.lineplot(df_attention_score_ts_mean_query, ax=axs, palette=color_palette, linewidth=2)

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)

axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")


axs.legend(
    # handles=handles,
    # ["Gated Transformer Network", "SAnD", "LSTM"],
    # title="Model",
    # facecolor="w",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=1,
    frameon=False,
    fancybox=True,
    shadow=True,
    #handleheight=1,
    #handlelength=3,
    prop=font_annots,
)

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)

# axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
# axs.xaxis.set_major_formatter(five_min_hour_formatter)

axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)

axs.set_xlabel("Time [h]")


axs.set_ylabel("Attention Score",
               )

axs.margins(x=0)

plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig_seq_attn_mean_line.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_temporal_features_averaged_lineplot.svg"),
                format="svg")
    fig_seq_attn_mean_line.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_temporal_features_averaged_lineplot.png"),
                format="png", dpi=300)
        


In [ ]:
attention_ts_feature_weighted = np.outer(attention_score_ts_mean_query, attention_score_channel_mean_query)
attention_ts_feature_weighted.shape


In [ ]:
df_attention_ts_feature_weighted = pd.DataFrame(data=attention_ts_feature_weighted,
                                                columns=sepsis_pred_data.considered_features)
# df_attention_ts_feature_weighted = VariableNames.get_pretty_dataframe(df=df_attention_ts_feature_weighted)
# df_attention_ts_feature_weighted = VariableNames.sort_dataframe(df_attention_ts_feature_weighted)


In [ ]:
fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=14, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)
df_attention_ts_feature_weighted_present = df_attention_ts_feature_weighted.loc[:, df_sample_ts_present.columns]
df_attention_ts_feature_weighted_present_pretty = VariableNames.get_pretty_dataframe(df_attention_ts_feature_weighted_present)
df_attention_ts_feature_weighted_present_pretty = VariableNames.sort_dataframe(df_attention_ts_feature_weighted_present_pretty)


# sns.heatmap(df_attention_ts_feature_weighted.T, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
sns.heatmap(df_attention_ts_feature_weighted_present_pretty.T, ax=axs, square=False, cbar=True,
            cbar_kws={"label": "Attention Score", "shrink": 1},
            xticklabels=1, yticklabels=1,
            cmap=sns.color_palette("magma", as_cmap=True))

# axs.invert_xaxis()
# axs.invert_yaxis()


axs.set_ylabel("Time Series Features",
               )

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)

axs.set_xlabel("Time [h]")
axs.margins(x=0)

# axs.set_xticklabels(axs.get_xticklabels(), fontsize=4, rotation=90)
axs.set_xticklabels(axs.get_xticklabels(), rotation=0)
axs.set_yticklabels(axs.get_yticklabels(),
                    fontsize=7
                    )

# Adjust layout
plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_temporal_features_outerproduct.svg"),
                format="svg")
    fig.savefig(plot_output_dir / Path(f"sample_{sample_index}_gtn_attention_temporal_features_outerproduct.png"),
            format="png", dpi=300)
